# RoadScan — train the pothole/crack detector

Runs on Colab's free T4. Training locally is not realistic: the dev machine is
CPU-only (`torch.cuda.is_available()` is `False`), and a nano model over ~13k
images for 100 epochs would take days there versus roughly an hour here.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

## Why this rebuilds the dataset instead of uploading it

The prepared dataset is ~818 MB. Uploading that over a slow domestic link is
painful, and pointless when Colab can fetch the same sources in a couple of
minutes on a datacentre connection. Only the finished model — about 6 MB —
comes back down to you.

Every cleaning step runs here exactly as it does locally, using the same
scripts, so the dataset this produces is the one that was verified:
no duplicate pictures, nothing straddling train/val, ~20% negatives.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install

`ultralytics>=8.4.83` matters: that is the release where the standalone
`tflite` export format was removed in favour of `format="litert"`, which is
what the export step calls. An older version fails at export, after training.

In [ ]:
!pip -q install 'ultralytics>=8.4.83' roboflow pillow
import ultralytics; print('ultralytics', ultralytics.__version__)

## 3. Upload the project scripts

Upload these four small files from the repo (a few tens of KB in total).
Select all four at once in the file picker:

- `ml/train_export.py`
- `tool/fetch_rdd.py`
- `tool/clean_dataset.py`
- `tool/check_dataset.py`

They are uploaded rather than cloned because the project is not a git
repository.

In [ ]:
import os, pathlib
from google.colab import files

os.makedirs('/content/roadscan/ml', exist_ok=True)
os.makedirs('/content/roadscan/tool', exist_ok=True)
os.chdir('/content/roadscan')

print('Select all four: train_export.py, fetch_rdd.py, clean_dataset.py, check_dataset.py')
up = files.upload()
for name in up:
    dest = 'ml' if name == 'train_export.py' else 'tool'
    pathlib.Path(dest, name).write_bytes(up[name])
    print('->', dest + '/' + name)

missing = [f for f in ('ml/train_export.py', 'tool/fetch_rdd.py',
                       'tool/clean_dataset.py', 'tool/check_dataset.py')
           if not pathlib.Path(f).exists()]
assert not missing, f'still missing: {missing} -- rerun this cell'
print('\nall four present')

## 4. Source A — RDD2022 India

Pulls just India (502 MiB) out of the 12.35 GB archive using HTTP range
requests. This is also where the negatives come from: 3,921 of India's 7,706
annotated images are explicitly annotated as damage-free road — real Indian
carriageway, which is worth far more here than foreign dashcam footage.

In [ ]:
!python tool/fetch_rdd.py --country India --extract --out /content/rdd2022

## 5. Source B — the Roboflow pothole set

Get a free API key at <https://app.roboflow.com> → Settings → API keys.

The copy Roboflow serves is the *raw* one, so it still contains the
duplicates: roughly 9,200 files holding only ~7,100 distinct pictures, with a
third of its valid/test set also present in train. The next cell fixes that.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))
ds = (rf.workspace('smartathon')
        .project('new-pothole-detection')
        .version(2)
        .download('yolov8', location='/content/photos'))
print('downloaded to', ds.location)

## 6. Clean it

Deduplicates by image content — filenames are useless, since Roboflow renames
everything to `<original>.rf.<hash>` and re-encodes — then re-splits so every
copy of a picture lands in the same split. Without this, validation mAP is
largely the model recognising pictures it was fitted on.

In [ ]:
!python tool/clean_dataset.py /content/photos --apply

## 7. Build the training set

Merges both sources onto the app's two classes, caps the crack imbalance,
adds the damage-free frames as negatives at 25% of the positives, then makes
a final pass to remove pictures the two sources share.

Check the `roboflow classes:` line it prints — it should list exactly
`['Pothole']`. Anything longer means the dataset yaml was misread.

In [ ]:
!python ml/train_export.py prepare \
    --roboflow /content/photos \
    --rdd2022 /content/rdd2022/India

## 8. Verify before spending GPU time

Checks every label rather than a sample, and applies the same rules
Ultralytics enforces at the start of training — so a fault surfaces here, in
seconds, rather than after the download and setup are already done.

It is not theoretical: the Roboflow export contains over a thousand polygon
segmentation rows, and Ultralytics rejects an entire detection dataset on
those ("labels require 5 columns each"). The prepare step converts them to
bounding boxes; this is what proves it did.

**Must print `OK -- no errors. Safe to train.` before you continue.**

In [ ]:
!python tool/check_dataset.py ml/dataset

## 9. Train

`yolo11n`: nano, because this runs on a phone. 640px is kept deliberately
even though a smaller input would be faster — 41% of the boxes are under 1%
of the frame, and shrinking the input is exactly what destroys recall on
those.

Augmentation is tuned for how the app actually captures: `hsv_v` for time of
day and shade, `degrees`/`scale` for a handheld phone, `fliplr` because a
pothole has no left/right meaning. `flipud=0` on purpose — an upside-down
road is not a thing the model will ever be shown.

Expect roughly 45–75 minutes on a T4. Keep the tab open; if Colab
disconnects, rerun this cell.

In [ ]:
!python ml/train_export.py train \
    --weights yolo11n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 32

## 10. Where to set the confidence threshold

The app currently hardcodes `AppConfig.minConfidence = 0.35`, which was a
guess made before any model existed. This measures it instead.

Two different thresholds are worth reading off the curve, because the app
makes two different decisions:

- a **high** one for silently accepting an upload, where a false positive
  puts a hazard on the map that is not there;
- a **lower** one for "we think we see something, please confirm", which is
  what keeps a genuine report from being rejected outright.

No threshold removes the third case — a real broken road the model simply
misses — which is why the manual-review escalation has to exist.

In [ ]:
from ultralytics import YOLO
m = YOLO('ml/runs/roadscan/weights/best.pt')
r = m.val(data='ml/dataset/roadscan.yaml', split='val')
print('mAP50    ', round(float(r.box.map50), 4))
print('mAP50-95 ', round(float(r.box.map), 4))
for i, name in enumerate(m.names.values()):
    try:
        print(f'  {name:8s} mAP50={float(r.box.ap50[i]):.4f}')
    except Exception:
        pass
print('\nsee runs/.../val*/ for the P-R and F1-vs-confidence curves')

## 11. Export for the phone

LiteRT with INT8, which roughly quarters the file and speeds up the CPU
delegate. INT8 calibrates against real images from `data` — without that the
quantisation ranges are guesses and accuracy drops far more than it should.

In [ ]:
!python ml/train_export.py export --imgsz 640 --int8

## 12. Download the model

Drop the file into `assets/models/roadscan.tflite` in the repo. `assets/models/`
is already declared in `pubspec.yaml`, so a rebuild picks it up and
`DetectionService` stops falling back to the stock COCO model.

Check the class names survived the export: the Flutter side maps them by
substring, so anything containing "pothole" or "crack" works and anything
else is silently dropped at runtime.

In [ ]:
from ultralytics import YOLO
print('class names:', YOLO('ml/runs/roadscan/weights/best.pt').names)

import os
from google.colab import files
p = 'assets/models/roadscan.tflite'
print(p, os.path.getsize(p) / 1e6, 'MB')
files.download(p)